In [0]:
# %run /Workspace/Users/marcoaurelioreislima@gmail.com/databricks-playground/projects/data_generator/CDC/DDLs

In [0]:
from pyspark.sql.functions import col, expr, date_add

PATH_CHECKPOINTS = "/Volumes/prd/streaming_management/checkpoints"
PATH_INPUT_CDC = "/Volumes/prd/stage/raw_data_managed/CDC/customers/json"

TABLE_CUSTOMER_BRONZE = "prd.l_bronze.customers_cdc"
TABLE_CUSTOMER_SILVER = "prd.l_silver.customers"

df_bronze_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(spark.table(TABLE_CUSTOMER_BRONZE).schema)
        .load(PATH_INPUT_CDC)
    .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{PATH_CHECKPOINTS}/bronze/customers_cdc")
        .trigger(availableNow=True)
        .table(TABLE_CUSTOMER_BRONZE)
        .awaitTermination()
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def upsert_to_delta(microBatchDF, batchId):
    window_1 = Window.partitionBy("user_id").orderBy(col("updated_at").asc())
    _ = (
        microBatchDF
            .filter(col("operation") == "INSERT")
            .withColumn("rank", row_number().over(window_1))
            .filter(col("rank") == 1).drop("rank")
            .createOrReplaceTempView("inserts")
    )

    spark.sql(f"""
    MERGE INTO prd.l_silver.customers s
    USING inserts i
    ON s.user_id = i.user_id
    WHEN NOT MATCHED THEN INSERT *
    """)

    window = Window.partitionBy("user_id").orderBy(col("operation"), col("updated_at").desc())
    _ = (
        microBatchDF
            .filter(col("operation") != "INSERT")
            .withColumn("rank", row_number().over(window))
            .filter(col("rank") == 1).drop("rank")
            .createOrReplaceTempView("updates")
    )

    spark.sql(f"""
    MERGE INTO prd.l_silver.customers s
    USING updates u
    ON s.user_id = u.user_id
    WHEN MATCHED AND u.operation = "DELETE" THEN DELETE
    WHEN MATCHED AND u.operation = "UPDATE" AND u.user_type IS NOT NULL THEN UPDATE SET s.user_type = u.user_type
    WHEN MATCHED AND u.operation = "UPDATE" AND u.first_name IS NOT NULL THEN UPDATE SET s.first_name = u.first_name
    WHEN MATCHED AND u.operation = "UPDATE" AND u.last_name IS NOT NULL THEN UPDATE SET s.last_name = u.last_name
    WHEN MATCHED AND u.operation = "UPDATE" AND u.income IS NOT NULL THEN UPDATE SET s.income = u.income
    WHEN MATCHED AND u.operation = "UPDATE" AND u.balance IS NOT NULL THEN UPDATE SET s.balance = u.balance
    WHEN MATCHED AND u.operation = "UPDATE" AND u.profession IS NOT NULL THEN UPDATE SET s.profession = u.profession
    WHEN MATCHED AND u.operation = "UPDATE" AND u.birth_date IS NOT NULL THEN UPDATE SET s.birth_date = u.birth_date
    WHEN MATCHED AND u.operation = "UPDATE" AND u.signup_date IS NOT NULL THEN UPDATE SET s.signup_date = u.signup_date
    """)
    


df_silver = (
    spark
        .readStream
        .table(TABLE_CUSTOMER_BRONZE)
        .writeStream
        .foreachBatch(upsert_to_delta)
        .option("checkpointLocation", f"{PATH_CHECKPOINTS}/silver/customers")
        .trigger(availableNow=True)
        .start()
        .awaitTermination()

)

In [0]:
def check():
    df_bronze = spark.table(TABLE_CUSTOMER_BRONZE)
    inserted = df_bronze.filter(f"operation = 'INSERT'").count()
    updated = df_bronze.filter(f"operation = 'UPDATE'").count()
    deleted = df_bronze.filter(f"operation = 'DELETE'").count()
    rows_silver = spark.table(TABLE_CUSTOMER_SILVER).count()
    print(f"inserted - deleted == {inserted - deleted}")
    print(f"rows_silver : {rows_silver}")



check()


In [0]:
df = (
    spark.read
        .format("delta")
        .option("readChangeData", True)
        .option("startingVersion", 2)
        .table(TABLE_CUSTOMER_SILVER)
)

df.display()

In [0]:
# %sql

# APPLY CHANGES INTO STREAM target
# FROM STREAM source
# KEYS (user_id)
# APPLY AS DELETE WHEN operation = "DELETE"
# SEQUENCE timestamp_datetime
# COLUMNS * EXCEPT (timestamp, _rescued_data, operation)
# STORED AS SCD TYPE 2